# データ概要

DB に蓄積されているデータの件数・期間・欠損状況をざっと確認するためのノートブック。
新しい分析を始めるときはまずこれを実行して、データの現状を把握する。

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from analysis.db import query_df
from analysis.plotting import setup_japanese_font

sns.set_theme(style="whitegrid")
setup_japanese_font()
pd.set_option("display.max_columns", None)

## テーブルごとの件数

In [ ]:
tables = ["horses", "jockeys", "trainers", "races", "race_results", "payoffs", "pre_race_odds"]
counts = query_df(
    " UNION ALL ".join(f"SELECT '{t}' AS table_name, COUNT(*) AS n_rows FROM {t}" for t in tables)
)
counts

## races の収録期間・年別件数

In [ ]:
races_per_year = query_df(
    """
    SELECT LEFT(date, 4) AS year, COUNT(*) AS n_races
    FROM races
    WHERE date IS NOT NULL
    GROUP BY 1
    ORDER BY 1
    """
)
races_per_year.plot.bar(x="year", y="n_races", figsize=(12, 4), legend=False)
plt.ylabel("n_races")
plt.title("年別レース数")
plt.tight_layout()

## race_results の主要カラムの欠損率

In [ ]:
race_results = query_df(
    """
    SELECT finishing_position, weight_carried, jockey_id, trainer_id,
           odds, popularity, horse_weight, last_3f
    FROM race_results
    """
)
(race_results.isna().mean().sort_values(ascending=False) * 100).round(2).rename("null_rate_%")

## 人気・オッズの分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
popularity_numeric = pd.to_numeric(race_results["popularity"], errors="coerce")
popularity_numeric.value_counts().sort_index().plot.bar(ax=axes[0])
axes[0].set_title("人気の分布")

odds_numeric = pd.to_numeric(race_results["odds"], errors="coerce")
odds_numeric.clip(upper=50).hist(bins=50, ax=axes[1])
axes[1].set_title("単勝オッズの分布（50倍でクリップ）")
plt.tight_layout()